In [5]:
import pygsti
import itertools
from pygsti.extras import ml
import numpy as np
from pygsti.circuits import Circuit
from pygsti.baseobjs import Label as L, Basis, QubitSpace
from pygsti.processors.processorspec import QubitProcessorSpec as QPS
from matplotlib import pyplot as plt
from pygsti.algorithms.randomcircuit import create_random_circuit as random_circuit
import tensorflow as tf
%load_ext autoreload
%autoreload 2

import itertools as _itertools
import copy as _copy

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
def up_to_weight_k_paulis(k: int, n: int):
    """
    Return all n-qubit Pauli strings with weight 1..k (non-identity count). If k=0, returns the all-identity string. If k>n, automatically sets k=n.

    A Pauli string is a length-n string over {'I','X','Y','Z'}.
    The *weight* is the number of non-'I' characters.

    This implementation is general for any k (clipped to n). It follows the same
    reverse-index convention as the original codebase/documentation:
    qubit i corresponds to string position (n - 1 - i). In other words, qubit 0
    is the *rightmost* character of the string.

    Parameters
    ----------
    k : int
        Positive integer. Maximum Pauli weight. Values > n are treated as n.
    n : int
        Number of qubits (string length).

    Returns
    -------
    list[str]
        All Pauli strings of weight 1..k.
    """

    if not isinstance(k, int) or k < 1:
        raise TypeError("Pauli weight must be an integer > 1.")
    
    if not isinstance(n, int) or n < 0:
        raise TypeError("Number of qubits must be a non-negative integer.")

    # Use a mutable "template" list of characters for efficient updates.
    # We will copy this list and then overwrite selected positions with X/Y/Z.
    base = list("I" * n)

    # Weight cannot exceed the number of qubits.
    if k > n:
        print("Pauli weight cannot exceed the number of qubits. Automatically setting k = min(k,n)")

    k = min(k, n)

    # These are indices into the Pauli string (0..n-1).
    #
    # Important convention note:
    #   If you interpret "qubit i" as mapping to string position (n-1-i),
    #   then the *rightmost* character corresponds to qubit 0.
    #
    # Here we simply generate strings by directly setting string indices.
    # Whether index j corresponds to qubit j or qubit (n-1-j) depends on
    # how the rest of your code interprets the string; this function just
    # produces strings with characters at indices 0..n-1.
    positions = list(range(n))

    # Collect all generated Pauli strings.
    paulis = []

    # Enumerate all weights w = 1..k
    for w in range(1, k + 1):

        # Choose which w positions (string indices) will be non-identity.
        # "support" is a tuple of length w with strictly increasing indices.
        for support in _itertools.combinations(positions, w):

            # For those w positions, choose an assignment of X/Y/Z at each position.
            # There are 3^w such assignments.
            for letters in _itertools.product("XYZ", repeat=w):

                # Copy the all-identity template, then place X/Y/Z on the support.
                s = base[:]  # shallow copy of list of characters
                for idx, P in zip(support, letters):
                    s[idx] = P

                # Convert list of characters back to a string and store.
                paulis.append("".join(s))

    return paulis

def up_to_weight_k_paulis_from_qubit_graph_old(k: int, n: int, qubit_graph_laplacian: np.array, num_hops: int) -> list:
    """Enumerate Pauli strings of weight 1..k (i.e., all Paulis contain at least one and
    at most k non-identity Paulis) with support on qubits connected by m 
    hops in the qubit connectivity graph.

    Weight-1 Paulis are always included. Weight-2 Paulis are included only when their
    two non-identity qubits are within `num_hops` steps as determined from powers of the
    (provided) graph Laplacian.

    Parameters
    ----------
    k : int
        Maximum Pauli weight (only `k <= 2` supported).
    n : int
        Number of qubits.
    qubit_graph_laplacian : numpy.ndarray
        Laplacian matrix of the qubit connectivity graph.
    num_hops : int
        Hop distance defining which qubit pairs are considered "close enough."

    Returns
    -------
    list[str]
        Pauli strings of length `n`. (Ordering uses reverse indexing convention in this file.) [qubit n, qubit n-1, ..., qubit 0]

    Notes
    -----
    Assumes the graph is connected (not strictly enforced by code).
    """
    assert (k <= 2), "Only implemented up to k = 2!"
    paulis = []

    # weight 1
    for i in range(n):
        for p in ['X', 'Y', 'Z']:
            nq_pauli = n * ['I']
            nq_pauli[n - 1 - i] = p # reverse indexing
            paulis.append(''.join(nq_pauli))
    
    # weight 2
    if k > 1:
        qubit_graph_laplacian = _copy.deepcopy(qubit_graph_laplacian) # Don't delete! Otherwise this function modifies the laplacian globally for some reason?
        laplace_power = np.linalg.matrix_power(qubit_graph_laplacian, num_hops)
        for i in np.arange(n):
            laplace_power[i, i] = 0
        # assert (laplace_power == 0).all(axis=1).any() == False, 'Graph must be connected'
    
        nodes_within_hops = []
        for i in range(n):
            nodes_within_hops.append(np.arange(n)[abs(laplace_power[i, :]) > 0])
    
        for i , qubit_list in enumerate(nodes_within_hops):
            unseen_qubits = qubit_list[np.where(qubit_list > i)[0]]
            for j in unseen_qubits:
                for p in ['X', 'Y', 'Z']:
                        for q in ['X', 'Y', 'Z']:
                            nq_pauli = n * ['I']
                            nq_pauli[n - 1 - i] = p # reverse indexing
                            nq_pauli[n - 1 - j] = q # reverse indexing
                            paulis.append(''.join(nq_pauli))
    
    return paulis


def up_to_weight_k_paulis_from_qubit_graph(
    k: int, n: int, qubit_graph_laplacian: np.ndarray, num_hops: int
) -> list[str]:
    """
    Enumerate Pauli strings of weight 1..k whose *support set* of non-identity qubits
    is connected under a "within num_hops" connectivity rule derived from the qubit graph.

    Convention used throughout this module:
        qubit i  <->  string position (n - 1 - i)
    so qubit 0 is the rightmost character in the Pauli string.

    Notes
    -----
    This is a generalization of the legacy k<=2 implementation:
      - For w=1, every single-qubit Pauli is allowed.
      - For w=2, a pair (i,j) is allowed iff i and j are within num_hops hops.
      - For w>2, a support S is allowed iff the induced subgraph on S is connected
        using the "within num_hops" adjacency.
    """
    # ---- Basic input checks ----
    if not isinstance(k, int) or k < 1:
        raise TypeError("Pauli weight, k, must be an integer > 1.")
    if not isinstance(n, int) or n < 0:
        raise TypeError("n must be a non-negative integer.")

    if k > n:
        print("Pauli weight cannot exceed the number of qubits. Automatically setting k = min(k,n)")

    # Clamp k so we never ask for weight > n.
    k = min(k, n)

    # Make a private copy to avoid mutating the caller's matrix accidentally.
    L = np.array(qubit_graph_laplacian, copy=True)

    # Sanity-check dimensions: Laplacian must be n x n.
    if L.shape != (n, n):
        raise ValueError("qubit_graph_laplacian must have shape (n, n).")

    # ---- Build the "within num_hops" connectivity relation ----
    #
    # Legacy code used:
    #   laplace_power = L**num_hops
    #   nodes are considered "within hops" if laplace_power[i,j] != 0
    #
    M = np.linalg.matrix_power(L, num_hops)

    # Remove diagonal entries; we don't want to treat i as "connected to itself"
    # for the purpose of defining edges between *distinct* qubits.
    np.fill_diagonal(M, 0)

    # Convert to a boolean adjacency: close[i,j] == True means "i and j are within num_hops"
    # according to the matrix-power criterion above.
    close = (np.abs(M) > 0)

    # If the underlying graph is undirected, "close" should be symmetric.
    # This line enforces symmetry just in case numerical issues or input oddities break it.
    close = np.logical_or(close, close.T)

    # ---- Helper: test whether a proposed support set is connected ----
    def support_is_connected(support_qubits: tuple[int, ...]) -> bool:
        """
        Return True iff the induced subgraph on `support_qubits` is connected,
        where edges are given by `close`.
        """
        # Empty support shouldn't appear here (we generate weights starting at 1),
        # and a single vertex is trivially connected.
        if len(support_qubits) <= 1:
            return True

        # We'll do a simple DFS/BFS over the support set using the `close` adjacency.
        support = list(support_qubits)
        support_set = set(support)

        # Start from the first qubit in the support.
        seen = {support[0]}
        stack = [support[0]]

        # Standard depth-first traversal restricted to nodes in the support.
        while stack:
            u = stack.pop()

            # Consider only neighbors v that are:
            #   (1) in the support set
            #   (2) adjacent to u under "close"
            #   (3) not yet visited
            for v in support:
                if v not in seen and close[u, v]:
                    seen.add(v)
                    stack.append(v)

        # Connected iff we reached every vertex in the support.
        return len(seen) == len(support_set)

    # ---- Enumerate all valid Pauli strings ----
    base = ['I'] * n            # template for fast construction
    paulis = []                 # output list of Pauli strings
    qubits = range(n)           # qubit labels 0..n-1

    # Enumerate all weights w = 1..k
    for w in range(1, k + 1):

        # Enumerate all possible supports (sets of qubits) of size w.
        for support_qubits in _itertools.combinations(qubits, w):

            # Skip supports that are not connected under the within-num_hops rule.
            if not support_is_connected(support_qubits):
                continue

            # For each support, assign an X/Y/Z to each qubit in the support.
            # There are 3^w assignments.
            for letters in _itertools.product("XYZ", repeat=w):

                # Start from all-identity string, then overwrite the support positions.
                s = base[:]

                # Place each chosen letter at the appropriate *string* index.
                # Remember: qubit q corresponds to string position (n-1-q).
                for q, P in zip(support_qubits, letters):
                    s[n - 1 - q] = P

                # Convert list-of-chars to a string and store it.
                paulis.append("".join(s))

    return paulis

In [9]:
def test_compare_up_to_weight_k_paulis_from_qubit_graph():
    """
    Compare a "new/general" implementation of up_to_weight_k_paulis_from_qubit_graph
    against the legacy implementation (which only supports k<=2) over a variety of inputs.

    How to use
    ----------
    1) Keep the legacy function accessible under a different name, e.g.
         up_to_weight_k_paulis_from_qubit_graph_old = up_to_weight_k_paulis_from_qubit_graph
       before you overwrite it with the new version.

    2) Run this test after defining the new version.

    What this test checks
    ---------------------
    * Only compares k in {1,2} because the legacy code asserts k<=2 and (as written)
      does not include weight-0 identity.
    * Compares outputs as sets (ordering-independent).
    * Tries multiple graphs, sizes, hop counts.

    Important caveat
    ----------------
    The "legacy" implementation determines "within num_hops" using matrix powers of the Laplacian.
    The new implementation must use the *same criterion* for the test to pass (the version I
    provided does).
    """
    # --------------------
    # Helper: build Laplacians for a few standard undirected graphs
    # --------------------
    def line_graph_laplacian(n: int) -> np.ndarray:
        A = np.zeros((n, n), dtype=int)
        for i in range(n - 1):
            A[i, i + 1] = 1
            A[i + 1, i] = 1
        D = np.diag(A.sum(axis=1))
        return D - A

    def complete_graph_laplacian(n: int) -> np.ndarray:
        A = np.ones((n, n), dtype=int) - np.eye(n, dtype=int)
        D = np.diag(A.sum(axis=1))
        return D - A

    def star_graph_laplacian(n: int, center: int = 0) -> np.ndarray:
        A = np.zeros((n, n), dtype=int)
        for j in range(n):
            if j == center:
                continue
            A[center, j] = 1
            A[j, center] = 1
        D = np.diag(A.sum(axis=1))
        return D - A

    # --------------------
    # Helper: compare ordering-independent and provide useful diffs
    # --------------------
    def assert_same_outputs(new_out, old_out, context: str):
        new_set = set(new_out)
        old_set = set(old_out)
        if new_set != old_set:
            only_in_new = sorted(new_set - old_set)[:20]
            only_in_old = sorted(old_set - new_set)[:20]
            raise AssertionError(
                f"Mismatch for {context}\n"
                f"  len(new)={len(new_out)} len(old)={len(old_out)}\n"
                f"  only_in_new (first 20)={only_in_new}\n"
                f"  only_in_old (first 20)={only_in_old}\n"
            )

    # --------------------
    # Test grid
    # --------------------
    ns = [1, 2, 3, 4, 5, 6]
    ks = [1, 2]          # legacy function supports only k<=2 and excludes weight-0 identity
    hops_list = [1, 2, 3,4]

    laplacians = [
        ("line", line_graph_laplacian),
        ("complete", complete_graph_laplacian),
        ("star", star_graph_laplacian),
    ]

    # --------------------
    # Run tests
    # --------------------
    # NOTE: rename `up_to_weight_k_paulis_from_qubit_graph_old` if your legacy function
    # has a different name.
    for n in ns:
        for lap_name, lap_fn in laplacians:
            L = lap_fn(n)

            for k in ks:
                for num_hops in hops_list:
                    context = f"n={n}, graph={lap_name}, k={k}, hops={num_hops}"

                    old_out = up_to_weight_k_paulis_from_qubit_graph_old(k, n, L, num_hops)
                    new_out = up_to_weight_k_paulis_from_qubit_graph(k, n, L, num_hops)

                    assert_same_outputs(new_out, old_out, context)

    print("PASS: new and old up_to_weight_k_paulis_from_qubit_graph match for all tested inputs.")

In [12]:
def line_graph_laplacian(n: int) -> np.ndarray:
    A = np.zeros((n, n), dtype=int)
    for i in range(n - 1):
        A[i, i + 1] = 1
        A[i + 1, i] = 1
    D = np.diag(A.sum(axis=1))
    return D - A

In [10]:
test_compare_up_to_weight_k_paulis_from_qubit_graph()

PASS: new and old up_to_weight_k_paulis_from_qubit_graph match for all tested inputs.


In [37]:
np.linalg.matrix_power(line_graph_laplacian(3),2)

array([[ 2, -3,  1],
       [-3,  6, -3],
       [ 1, -3,  2]])

In [35]:
n = 3
L = line_graph_laplacian(n)
print(L)
print()
A = np.diag(np.diag(L))-L
print(A)
print()
print(np.linalg.matrix_power(A,3))

[[ 1 -1  0]
 [-1  2 -1]
 [ 0 -1  1]]

[[0 1 0]
 [1 0 1]
 [0 1 0]]

[[0 2 0]
 [2 0 2]
 [0 2 0]]


In [34]:
num_hops = 12
close = np.zeros((n, n), dtype=bool)
if num_hops > 0:
    Apow = A.copy()
    for _h in range(1, num_hops + 1):
        close |= (Apow > 0)
        Apow = Apow @ A  # next power
np.fill_diagonal(close, False)
print(close)


[[False  True  True]
 [ True False  True]
 [ True  True False]]


In [12]:
num_qubits = 4

gate_names = ['Gxpi2', 'Gypi2', 'Gcphase']

availability= {'Gcphase': [(0,1), (1,2), (2,3), (3,0)]}

qubit_labels = [i for i in range(num_qubits)]

pspec = QPS(num_qubits = num_qubits, qubit_labels=qubit_labels,
            gate_names=gate_names, availability=availability)

In [14]:
num_circs = 300
circuits = [random_circuit(pspec, np.random.randint(1, 30 + 1), sampler='edgegrab', samplerargs=[0.5]) for i in range (num_circs)]

In [15]:
def sample_errors_dict(scale_factor=1, crosstalk_scale_factor=1):

    error_rates_dict = {}
    
    def sample_local_oneQ_Hamiltonian_error():
        return scale_factor * 0.01 * 2 * (np.random.rand() - 0.5)
    
    def sample_local_oneQ_stochastic_error():
        return scale_factor * 0.001 * np.random.rand()
    
    def sample_neighbor_oneQ_Hamiltonian_error():
        return crosstalk_scale_factor * scale_factor * 0.005 * 2 * (np.random.rand() - 0.5)
    
    def sample_local_twoQ_Hamiltonian_error():
        return scale_factor * 0.01 * 2 * (np.random.rand() - 0.5)
    
    def sample_local_twoQ_stochastic_error():
        return scale_factor * 0.001  * np.random.rand()
    
    def sample_twoQ_Hamiltonian_ZZZZ_error():
        return crosstalk_scale_factor * scale_factor * 0.02  * np.random.rand()
    
    error_rates_dict[L('Gypi2',0)] = {('H','YIII:0,1,2,3'): sample_local_oneQ_Hamiltonian_error()}
    error_rates_dict[L('Gypi2',1)] = {('H','IYII:0,1,2,3'): sample_local_oneQ_Hamiltonian_error()}
    error_rates_dict[L('Gypi2',2)] = {('H','IIYI:0,1,2,3'): sample_local_oneQ_Hamiltonian_error()}
    error_rates_dict[L('Gypi2',3)] = {('H','IIIY:0,1,2,3'): sample_local_oneQ_Hamiltonian_error()}
    
    error_rates_dict[L('Gxpi2',0)] = {('H','XIII:0,1,2,3'): sample_local_oneQ_Hamiltonian_error()}
    error_rates_dict[L('Gxpi2',1)] = {('H','IXII:0,1,2,3'): sample_local_oneQ_Hamiltonian_error()}
    error_rates_dict[L('Gxpi2',2)] = {('H','IIXI:0,1,2,3'): sample_local_oneQ_Hamiltonian_error()}
    error_rates_dict[L('Gxpi2',3)] = {('H','IIIX:0,1,2,3'): sample_local_oneQ_Hamiltonian_error()}
    
    error_rates_dict[L('Gypi2',0)].update({('S','YIII:0,1,2,3'): sample_local_oneQ_stochastic_error()})
    error_rates_dict[L('Gypi2',1)].update({('S','IYII:0,1,2,3'): sample_local_oneQ_stochastic_error()})
    error_rates_dict[L('Gypi2',2)].update({('S','IIYI:0,1,2,3'): sample_local_oneQ_stochastic_error()})
    error_rates_dict[L('Gypi2',3)].update({('S','IIIY:0,1,2,3'): sample_local_oneQ_stochastic_error()})
    
    error_rates_dict[L('Gxpi2',0)].update({('S','XIII:0,1,2,3'): sample_local_oneQ_stochastic_error()})
    error_rates_dict[L('Gxpi2',1)].update({('S','IXII:0,1,2,3'): sample_local_oneQ_stochastic_error()})
    error_rates_dict[L('Gxpi2',2)].update({('S','IIXI:0,1,2,3'): sample_local_oneQ_stochastic_error()})
    error_rates_dict[L('Gxpi2',3)].update({('S','IIIX:0,1,2,3'): sample_local_oneQ_stochastic_error()})
    
    error_rates_dict[L('Gxpi2',0)].update({('H','IXII:0,1,2,3'): sample_neighbor_oneQ_Hamiltonian_error()})
    error_rates_dict[L('Gxpi2',0)].update({('H','IIIX:0,1,2,3'): sample_neighbor_oneQ_Hamiltonian_error()})
    error_rates_dict[L('Gxpi2',1)].update({('H','XIII:0,1,2,3'): sample_neighbor_oneQ_Hamiltonian_error()})
    error_rates_dict[L('Gxpi2',1)].update({('H','IIXI:0,1,2,3'): sample_neighbor_oneQ_Hamiltonian_error()})
    error_rates_dict[L('Gxpi2',2)].update({('H','IXII:0,1,2,3'): sample_neighbor_oneQ_Hamiltonian_error()})
    error_rates_dict[L('Gxpi2',2)].update({('H','IIIX:0,1,2,3'): sample_neighbor_oneQ_Hamiltonian_error()})
    error_rates_dict[L('Gxpi2',3)].update({('H','IIXI:0,1,2,3'): sample_neighbor_oneQ_Hamiltonian_error()})
    error_rates_dict[L('Gxpi2',3)].update({('H','XIII:0,1,2,3'): sample_neighbor_oneQ_Hamiltonian_error()})
    
    error_rates_dict[L('Gypi2',0)].update({('H','IYII:0,1,2,3'): sample_neighbor_oneQ_Hamiltonian_error()})
    error_rates_dict[L('Gypi2',0)].update({('H','IIIY:0,1,2,3'): sample_neighbor_oneQ_Hamiltonian_error()})
    error_rates_dict[L('Gypi2',1)].update({('H','YIII:0,1,2,3'): sample_neighbor_oneQ_Hamiltonian_error()})
    error_rates_dict[L('Gypi2',1)].update({('H','IIYI:0,1,2,3'): sample_neighbor_oneQ_Hamiltonian_error()})
    error_rates_dict[L('Gypi2',2)].update({('H','IYII:0,1,2,3'): sample_neighbor_oneQ_Hamiltonian_error()})
    error_rates_dict[L('Gypi2',2)].update({('H','IIIY:0,1,2,3'): sample_neighbor_oneQ_Hamiltonian_error()})
    error_rates_dict[L('Gypi2',3)].update({('H','IIYI:0,1,2,3'): sample_neighbor_oneQ_Hamiltonian_error()})
    error_rates_dict[L('Gypi2',3)].update({('H','YIII:0,1,2,3'): sample_neighbor_oneQ_Hamiltonian_error()})
    
    error_rates_dict[L('Gcphase',(0,1))] = {('H','ZZII:0,1,2,3'): sample_local_twoQ_Hamiltonian_error()}
    error_rates_dict[L('Gcphase',(0,1))].update({('H','ZIII:0,1,2,3'): sample_local_twoQ_Hamiltonian_error()})
    error_rates_dict[L('Gcphase',(0,1))].update({('H','IZII:0,1,2,3'): sample_local_twoQ_Hamiltonian_error()})
    error_rates_dict[L('Gcphase',(1,2))] = {('H','IZZI:0,1,2,3'): sample_local_twoQ_Hamiltonian_error()}
    error_rates_dict[L('Gcphase',(1,2))].update({('H','IZII:0,1,2,3'): sample_local_twoQ_Hamiltonian_error()})
    error_rates_dict[L('Gcphase',(1,2))].update({('H','IIZI:0,1,2,3'): sample_local_twoQ_Hamiltonian_error()})
    error_rates_dict[L('Gcphase',(2,3))] = {('H','IIZZ:0,1,2,3'): sample_local_twoQ_Hamiltonian_error()}
    error_rates_dict[L('Gcphase',(2,3))].update({('H','IIIZ:0,1,2,3'): sample_local_twoQ_Hamiltonian_error()})
    error_rates_dict[L('Gcphase',(2,3))].update({('H','IIZI:0,1,2,3'): sample_local_twoQ_Hamiltonian_error()})
    error_rates_dict[L('Gcphase',(3,0))] = {('H','ZIIZ:0,1,2,3'): sample_local_twoQ_Hamiltonian_error()}
    error_rates_dict[L('Gcphase',(3,0))].update({('H','ZIII:0,1,2,3'): sample_local_twoQ_Hamiltonian_error()})
    error_rates_dict[L('Gcphase',(3,0))].update({('H','IIIZ:0,1,2,3'): sample_local_twoQ_Hamiltonian_error()})
    
    error_rates_dict[L('Gcphase',(0,1))].update({('S','ZZII:0,1,2,3'): sample_local_twoQ_stochastic_error()})
    error_rates_dict[L('Gcphase',(0,1))].update({('S','ZIII:0,1,2,3'): sample_local_twoQ_stochastic_error()})
    error_rates_dict[L('Gcphase',(0,1))].update({('S','IZII:0,1,2,3'): sample_local_twoQ_stochastic_error()})
    error_rates_dict[L('Gcphase',(1,2))].update({('S','IZZI:0,1,2,3'): sample_local_twoQ_stochastic_error()})
    error_rates_dict[L('Gcphase',(1,2))].update({('S','IZII:0,1,2,3'): sample_local_twoQ_stochastic_error()})
    error_rates_dict[L('Gcphase',(1,2))].update({('S','IIZI:0,1,2,3'): sample_local_twoQ_stochastic_error()})
    error_rates_dict[L('Gcphase',(2,3))].update({('S','IIZZ:0,1,2,3'): sample_local_twoQ_stochastic_error()})
    error_rates_dict[L('Gcphase',(2,3))].update({('S','IIIZ:0,1,2,3'): sample_local_twoQ_stochastic_error()})
    error_rates_dict[L('Gcphase',(2,3))].update({('S','IIZI:0,1,2,3'): sample_local_twoQ_stochastic_error()})
    error_rates_dict[L('Gcphase',(3,0))].update({('S','ZIIZ:0,1,2,3'): sample_local_twoQ_stochastic_error()})
    error_rates_dict[L('Gcphase',(3,0))].update({('S','ZIII:0,1,2,3'): sample_local_twoQ_stochastic_error()})
    error_rates_dict[L('Gcphase',(3,0))].update({('S','IIIZ:0,1,2,3'): sample_local_twoQ_stochastic_error()})
    
    error_rates_dict[L('Gcphase',(0,1))].update({('H','ZZZZ:0,1,2,3'): sample_twoQ_Hamiltonian_ZZZZ_error()})
    error_rates_dict[L('Gcphase',(1,2))].update({('H','ZZZZ:0,1,2,3'): sample_twoQ_Hamiltonian_ZZZZ_error()})
    error_rates_dict[L('Gcphase',(2,3))].update({('H','ZZZZ:0,1,2,3'): sample_twoQ_Hamiltonian_ZZZZ_error()})
    error_rates_dict[L('Gcphase',(3,0))].update({('H','ZZZZ:0,1,2,3'): sample_twoQ_Hamiltonian_ZZZZ_error()})

    return error_rates_dict

In [16]:
error_rates_dict = sample_errors_dict()
error_model = pygsti.models.create_cloud_crosstalk_model(pspec, lindblad_error_coeffs=error_rates_dict,
                                                         errcomp_type="errorgens")

In [17]:
nbit_strings = [''.join(p) for p in itertools.product('01', repeat=num_qubits)]
simulated_probabilities = []
for i, circuit in enumerate(circuits):
    probs_dict =  error_model.probabilities(circuit)
    simulated_probabilities.append([probs_dict[bs] for bs in nbit_strings])
simulated_probabilities = np.array(simulated_probabilities)

In [18]:
# We're going to create a neural network with all the correct error generators, so
# this extracts them from the error_rates_dict.
modelled_error_generators  = []
for gate_error_rates in error_rates_dict.values():
    for error_label in gate_error_rates.keys():
        error_gen = (error_label[0], (error_label[1].split(':')[0],))
        if error_gen not in modelled_error_generators:
            modelled_error_generators.append(error_gen)
print(modelled_error_generators)
print(len(modelled_error_generators))

[('H', ('YIII',)), ('S', ('YIII',)), ('H', ('IYII',)), ('H', ('IIIY',)), ('S', ('IYII',)), ('H', ('IIYI',)), ('S', ('IIYI',)), ('S', ('IIIY',)), ('H', ('XIII',)), ('S', ('XIII',)), ('H', ('IXII',)), ('H', ('IIIX',)), ('S', ('IXII',)), ('H', ('IIXI',)), ('S', ('IIXI',)), ('S', ('IIIX',)), ('H', ('ZZII',)), ('H', ('ZIII',)), ('H', ('IZII',)), ('S', ('ZZII',)), ('S', ('ZIII',)), ('S', ('IZII',)), ('H', ('ZZZZ',)), ('H', ('IZZI',)), ('H', ('IIZI',)), ('S', ('IZZI',)), ('S', ('IIZI',)), ('H', ('IIZZ',)), ('H', ('IIIZ',)), ('S', ('IIZZ',)), ('S', ('IIIZ',)), ('H', ('ZIIZ',)), ('S', ('ZIIZ',))]
33


In [19]:
 set(np.array([2,2,3]))

{np.int64(2), np.int64(3)}

In [20]:
tensors = ml.encoding.error_generator_tensors(circuits, modelled_error_generators, pspec, alpha_representation='concise')
indices = tensors['indices'] # Not needed by the QPANN but are computed to compute the alphas and so still returned
signs = tensors['signs']  # Not needed by the QPANN but are computed to compute the alphas and so still returned
probabilities = tensors['probabilities'] 
alphas = tensors['alphas']

100%|██████████| 300/300 [05:52<00:00,  1.18s/it]


In [21]:
# encoder = ml.encoding.StandardCircuitEncoder(pspec)
adjacency_matrix = ml.snippers.undirected_adjacency_matrix_from_edges(availability['Gcphase'], qubit_labels)
snipper = ml.snippers.layer_snipper_from_qubit_graph(modelled_error_generators, encoder, adjacency_matrix, hops=1)
x = [circuits_tensor, alphas, probabilities]
qpann = ml.qpanns.QPANN(encoder.length, modelled_error_generators, snipper, probability_computation='concise')

NameError: name 'circuits_tensor' is not defined

In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
early_stopping = tf.keras.callbacks.EarlyStopping(
                monitor='val_R2Score',  # Monitor the validation loss
                patience=20,         # Number of epochs with no improvement after which training will be stopped
                restore_best_weights=True,  # Restore model weights from the epoch with the best validation loss
                verbose=1            # Verbosity mode, 1 = progress messages
            )

qpann.compile(optimizer, loss=tf.keras.losses.MSE, metrics=['R2Score', 'mae'])
qpann.summary()
# Fit the model using training data and include validation data
history = qpann.fit(
    x, 
    simulated_probabilities, 
    epochs=100,
    #validation_data=(x_new['validate'], y_split[sfs][nm_ind][num_shots]['validate']),  # Include validation data
    callbacks=[early_stopping], 
    verbose=1, 
    batch_size=100, 
    validation_batch_size=100,
    shuffle=True
)

fig = plt.figure(figsize=(7,4))
plt.plot(history.history['loss'], label='loss')